- datasets 라이브러리로부터 허깅페이스의 데이터를 다운로드 하는데 사용할 load_dataset을 임포트합니다.


In [1]:
from datasets import load_dataset

dataset = load_dataset("iamjoon/klue-mrc-ko-rag-dataset")

c:\workspace\python\rag_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 1884/1884 [00:00<00:00, 34071.36 examples/s]


- 허깅페이스로부터 데이터를 로드하고 이를 출력해보겠습니다.
- 로드된 데이터셋을 판다스 데이터프레임으로 변환합니다.
- 판다스로 변환한 이유는 데이터 조작과 분석을 보다 쉽게 하기 위함입니다.
- 실습에 필요한 열만 선택하여 새로운 데이터프레임을 만듭니다.


In [ ]:
df = dataset["train"].to_pandas()
df = df[["question", "search_result", "answer", "extracted_ref_numbers", "type"]]
df.head()

,question,search_result,answer,extracted_ref_numbers,type
0,북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?,[열대우림 '장마전선'은 벵골만과 서북태평양에서 동아시아 몬순의 하위시스템으로 조성...,북태평양 기단과 오호츠크해 기단이 만나 형성되는 장마전선은 한반도에 약 한 달가량 ...,[2],mrc_question
1,지능형 생산자동화 기반기술을 개발중인 스타트업은?,[부산시와 (재)부산정보산업진흥원(원장 이인숙)이 ‘2020~2021년 지역SW서비...,지능형 생산자동화 기반기술을 개발 중인 스타트업으로는 삼보테크놀로지가 있습니다. 삼...,[1],mrc_question
2,개막전에서 3안타 2실점을 기록해서 패한 선수는?,"[;한큐 - 사카모토 도시조\n* 타석에서의 상황 : 헛스윙, 볼, 헛스윙, 파울,...",개막전에서 3안타 2실점을 기록해서 패한 선수는 사이타마 세이부 라이온스의 와쿠이 ...,[2],mrc_question
3,컵라면 매출에서 불닭볶음면을 이긴 상품은?,[유명 맛집 이름을 달고 나온 편의점 자체상표(PB) 라면이 인기를 끌고 있다. ‘...,세븐일레븐에서 판매하는 '교동반점 짬뽕'이 삼양식품의 '불닭볶음면'을 제치고 컵라면...,[1],mrc_question
4,정부에게 환경과 관련해서 우선적으로 원조 받고 있는 곳은?,[정부가 전기자동차를 보급하는 방식을 바꾼다. 주로 일반 소비자와 관공서에 판매하던...,정부가 환경과 관련하여 우선적으로 원조를 받고 있는 곳은 주로 전기자동차 보급과 관...,[1],mrc_question


- question(질문), search_result(질문에 대한 검색 결과), answer(최종답변), extracted_ref_numers(답변에서 인용된 문서 번호의 리스트), type(데이터 유형)입니다.


In [3]:
print("데이터 타입 종류:", df["type"].unique())

데이터 타입 종류: ['mrc_question' 'mrc_question_with_1_to_4_negative' 'synthetic_question'
 'paraphrased_question' 'no_answer']


| type                              | 내용                                                                                                                                                                                                                                         |
| --------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| mrc_question                      | 장소나 이름, 날짜 등을 묻는 지엽적인 질문<br/>검색 결과(search_result)는 5개로 고정                                                                                                                                                          |
| mrc_question_with_1_to_4_negative | 장소나 이름, 날짜 등을 묻는 지엽적인 질문<br/>검색 결과(search_result)가 1~4개 사이인 유형                                                                                                                                                   |
| paraphrased_question              | 장소나 이름, 날짜 등을 묻는 지엽적인 질문이며 질문의 형태가 문장이 아닌 명사구의 형태<br/>검색 결과(search_result)는 5개로 고정                                                                                                              |
| synthetic_question                | 이유, 장점, 단점 등과 같은 포괄적인 질문<br/>검색 결과(search_result)는 5개로 고정<br/>포괄적인 질문이므로 일반적으로 다수의 문서를 인용하게 된다는 특징이 있음.<br/>따라서 일반적으로 인용 문서 번호(extracted_ref_numbers)의 값이 2개 이상 |
| no_answer                         | 질문에 대한 답이 검색 결과에 없는 데이터<br/>답변(answer)에는 검색 결과에 질문에 대한 답이 없다고 안내해야만 함<br />검색 결과(search_result)는 5개로 고정                                                                                   |


- 각 데이터 유형은 실제 검색 증강 생성에서 발생할 수 있는 다양한 시나리오에 대응하기 위한 목적으로 만들어졌습니다.
- 학습 후에 실제 상황에서 질문에 대한 답이 데이터에 없는 경우가 발생하면 성능이 저하되는 경우가 많습니다.
- 따라서 '지엽적인 질문', '검색 결과가 5개인 경우 또는 5개 미만인 경우', '지엽적인 질문이지만 질문의 형태가 명사구인 경우', '포괄적인 질문으로 답변 시 다수의 문서를 인용해야 하는 경우', '질문에 대한 답이 검색 결과에 없는 경우'와 같이 다양한 경우에 대비해야 합니다.
- 이렇게 학습데이터를 구성해서 학습해야만 실제 상황에서 발생할 수 있는 다양한 상황에 성능 하락없이 대응할 수 있습니다.
- 이제 각 데이터의 특징을 실제 출력을 통해 이해해 봅시다.
